In [ ]:
from pathlib import Path
import numpy as np, nibabel as nib
from tqdm import tqdm

TRAIN_DIR = Path("/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData")
valid_patients = [p for p in sorted(TRAIN_DIR.glob("BraTS20_Training_*"))
                  if (p / f"{p.name}_seg.nii").exists()]
print(len(valid_patients))

CACHE = Path("/kaggle/working/brats_cache"); CACHE.mkdir(exist_ok=True)
C0, C1 = 24, 216   # 192x192 center crop

def zscore(v):
    m = v > 0
    out = np.zeros_like(v, dtype=np.float32)
    if m.any():
        out[m] = (v[m] - v[m].mean()) / (v[m].std() + 1e-8)
    return out

for p in tqdm(valid_patients):
    n = p.name
    seg = nib.load(p / f"{n}_seg.nii").get_fdata().astype(np.uint8)
    seg[seg == 4] = 3
    lost = (seg > 0).sum() - (seg[C0:C1, C0:C1] > 0).sum()
    if lost: print(f"{n}: crop cuts {lost} tumor voxels")
    keep = [s for s in range(seg.shape[2]) if (seg[C0:C1, C0:C1, s] > 0).sum() >= 100]
    x = np.stack([
        zscore(nib.load(p / f"{n}_{m}.nii").get_fdata(dtype=np.float32))[C0:C1, C0:C1][:, :, keep]
        for m in ("t1", "t1ce", "t2", "flair")
    ])
    np.save(CACHE / f"{n}_x.npy", x.transpose(3, 0, 1, 2).astype(np.float16))
    np.save(CACHE / f"{n}_y.npy", seg[C0:C1, C0:C1][:, :, keep].transpose(2, 0, 1))

368


 93%|█████████▎| 343/368 [16:02<01:46,  4.25s/it]

BraTS20_Training_344: crop cuts 18 tumor voxels


100%|██████████| 368/368 [17:35<00:00,  2.87s/it]
